# optuna vs gridsearch

same xgboost tuning problem. timing both.


In [ ]:
import time
import optuna
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score
import xgboost as xgb

X, y = load_breast_cancer(return_X_y=True)


In [ ]:
def objective(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 2, 8),
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 0.3),
        'n_estimators': trial.suggest_int('n_estimators', 50, 400),
        'subsample': trial.suggest_uniform('subsample', 0.6, 1.0),
    }
    clf = xgb.XGBClassifier(**params)
    return cross_val_score(clf, X, y, cv=3, scoring='roc_auc').mean()


In [ ]:
t0 = time.time()
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)
opt_time = time.time() - t0
print('optuna best:', study.best_value, 'in', opt_time, 's')


In [ ]:
# grid search
from sklearn.model_selection import GridSearchCV
grid = {
    'max_depth':[2,4,6],
    'learning_rate':[0.01, 0.05, 0.1],
    'n_estimators':[100, 200, 400],
    'subsample':[0.7, 1.0],
}
t0 = time.time()
gs = GridSearchCV(xgb.XGBClassifier(), grid, cv=3, scoring='roc_auc')
gs.fit(X, y)
print('grid best:', gs.best_score_, 'in', time.time()-t0, 's')
